# 01 — Data acquisition

Download a regional NOAA OISST v2.1 subset through NCEI ERDDAP, with provenance and
structural validation.

The acquisition helpers live in `oisst_fno.data`, the manifest in `oisst_fno.provenance`,
and the structural checks in `oisst_fno.validation`. Only reusable logic sits in `src/`;
the summaries and plots below stay here.

**Dataset metadata**, verified against the live ERDDAP service for
`ncdc_oisst_v2_avhrr_by_time_zlev_lat_lon`:

| Property | Value |
|---|---|
| Product version | `Version v02r01` (OISST v2.1) |
| Processing level | NOAA Level 4 |
| Variables | `sst`, `anom`, `err`, `ice` |
| `sst` units | Celsius (stored `valid_min`/`valid_max` of -300/4500 in hundredths of a degree, i.e. -3 to 45 °C) |
| Grid | 0.25° in latitude and longitude |
| Longitude convention | `[0, 360)` |
| Latitude range | -89.875 to 89.875 |
| Dataset DOI | [10.25921/RE9P-PT57](https://doi.org/10.25921/RE9P-PT57) |

The exposed time coverage of this ERDDAP feature collection starts in 2020, which is why
the experiment begins there. **If NOAA changes the service, re-check this table before
rerunning the pipeline** — the constants in `oisst_fno` encode these values.

In [ ]:
from pathlib import Path

from oisst_fno.data import (
    OISST_DOI,
    OISST_PRODUCT_VERSION,
    Region,
    build_erddap_url,
    download_subset,
    open_oisst,
)
from oisst_fno.provenance import DownloadManifest, manifest_path_for
from oisst_fno.validation import validate_oisst_dataset

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
REGION = Region()

START_DATE = "2020-03-01"
END_DATE = "2026-07-31"
DESTINATION = RAW / f"oisst_{START_DATE}_{END_DATE}_ne_atlantic.nc"

print(f"product version: {OISST_PRODUCT_VERSION}")
print(f"dataset DOI    : {OISST_DOI}")
print(build_erddap_url(START_DATE, END_DATE, REGION, variables=("sst",)))

## Smoke test first

`smoke_test=True` fetches only the first few days. It exercises the whole path — URL
construction, network, atomic write, payload check, manifest, validation — against a few
megabytes, so a mistake in the bounds or dates surfaces in seconds rather than after a
multi-year download.

In [ ]:
smoke_path = download_subset(
    destination=RAW / "oisst_smoke_test.nc",
    start_date=START_DATE,
    end_date=END_DATE,
    region=REGION,
    variables=("sst",),
    smoke_test=True,
)

smoke_report = validate_oisst_dataset(open_oisst(smoke_path))
print("valid:", smoke_report.ok)
for key, value in smoke_report.summary.items():
    print(f"  {key}: {value}")
for issue in smoke_report.issues:
    print(f"  ISSUE {issue}")

## Full download

Network call, kept explicit rather than hidden inside model code. Transient failures are
retried with bounded exponential backoff; a client error such as a bad date range fails
immediately, because retrying cannot fix it.

The file is written to a `.part` path and moved into place only once the body is
complete, the declared length matches, and the payload really is NetCDF — so an
interrupted download can never be mistaken for complete data.

In [ ]:
path = download_subset(
    destination=DESTINATION,
    start_date=START_DATE,
    end_date=END_DATE,
    region=REGION,
    variables=("sst",),
)
path

## Provenance

The sidecar manifest answers "what source data produced this?" without needing to trust
memory or a filename. `open_oisst` verifies it automatically, so a truncated or edited
file raises rather than loading.

In [ ]:
manifest = DownloadManifest.read(manifest_path_for(path))

for key, value in manifest.to_dict().items():
    print(f"{key:>16}: {value}")

## Structural validation

Coordinates, 0.25° spacing, daily continuity, duplicated timestamps, value range, and
land-mask stability. Failures are reported explicitly rather than silently repaired —
a moving land mask, for example, would quietly change which cells the masked metrics
average over.

In [ ]:
ds = open_oisst(path)
report = validate_oisst_dataset(ds)

for key, value in report.summary.items():
    print(f"{key:>12}: {value}")

if report.ok:
    print("
All structural checks passed.")
else:
    for issue in report.issues:
        print(f"
ISSUE {issue}")

# Stop the notebook here if the subset is not usable.
report.raise_for_status()
ds

Preprocessing is **not** fitted here. Normalization is estimated on the training split
only, in notebook `04`, so no validation or test information can leak into it.